In [25]:
import os
import sys
import time
import yaml
import pandas as pd
import numpy as np
import json

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import llm_tools as llt


In [26]:
MODEL = "claude-opus-4-6"
LLM_PROVIDER = "claude" # openai | claude
input_cost = 2.50 # batch cost per 1M tokens
output_cost = 12.50 # batch cost per 1M tokens
output_tokens_per_request = 1000
REQUESTS_PER_BATCH = 4000

In [27]:
MIN_IDX = 0
MAX_IDX = 0
REPLACE = False
REPLACE_OUTPUT_FILE = True
MODE = 'WRITE'  # ESTIMATE | BATCH | WRITE

In [28]:
meetings_df = pd.read_csv(os.path.join(DATA_PATH, "intermediate_data/cpc/meetings-manifest.csv"))
DATES = sorted(list(meetings_df['date']))

In [29]:
PROMPT = r"""
The following extracted PDF text contains the agenda for a LA City Planning Commission meeting. 

Read each agenda item and extract its agenda item number, title (which is often just a Planning Department case number), and a short summary of the agenda item.

Return a JSON in the following format:

{
    "agenda_items": [
        {
            "item_no": "<agenda item number>",
            "title": "<title of agenda item>",
            "address": "<address or location of agenda item if applicable, empty string if not>",
            "council_district": "<council district number as integer if applicable, empty string if not>",
            "consent_calendar": "<whether item is on consent calendar: yes|no>",
            "is_appeal": "<whether item is an appeal of a previous decision: yes|no|n/a>",
            "summary": "<short summary of agenda item>"
        }
    ]
}

Guidelines:
- Return valid JSON only, no other text or markdown formatting.
- Agenda item numbers are usually formatted as "1.", "2.", "3a.", "3b.", etc.
- Agenda titles are often Planning Department case numbers like "CPC-2023-1234-DB-SPR", or generic matters of business like "Director's Report and Commission Business" or "General Public Comment".
- When the agenda item is a planning case, focus the summary on the key aspects of the case, such as the project type, location, and specific requests being made to the Commission.

AGENDA:

"""

In [30]:
# write prompt to figures
with open(os.path.join(LOCAL_PATH, 'figures', 'agenda_split_prompt.tex'), 'w') as f:
    out = PROMPT + "<<agenda text>>"
    out = out.strip()
    out = out.replace('\n', '\\\\ \n')
    f.write(out)

In [31]:
input_tokens = 0
output_tokens = 0
n_requests = 0
n_direct = 0
batch_num = 0

rel_path = os.path.join(DATA_PATH, "response_store")
batch_db_path = os.path.join(rel_path, "batch_jobs.db")
response_db_path = os.path.join(rel_path, "responses.db")

batch_db_conn = llt.get_batch_jobs_db_conn(batch_db_path)
response_db_conn = llt.get_response_store_db_conn(response_db_path)

t0 = time.time()
prompts_to_submit = []
for idx, row in meetings_df.iterrows():
    if idx < MIN_IDX or idx > MAX_IDX:
        continue

    date = row['date']
    year = date[0:4]

    print(f"{idx} {date} ...")

    # Input data
    input_filepath = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date, "agenda.txt")
    override_filepath = os.path.join(LOCAL_PATH, "intermediate_data/cpc", year, date, "agenda-manual-override.txt")
    if (not os.path.exists(input_filepath)) and (not os.path.exists(override_filepath)):
        print("No agenda text file found, skipping.")
        continue
    if os.path.exists(override_filepath):
        with open(override_filepath, 'r', encoding='utf-8') as f:
            text = f.read()
    else:
        with open(input_filepath, 'r', encoding='utf-8') as f:
            text = f.read()

    # Output paths
    output_dir = os.path.join(DATA_PATH, "intermediate_data/cpc", year, date)
    os.makedirs(output_dir, exist_ok=True)
    output_filepath = os.path.join(output_dir, "agenda-summary.json")
    if (not REPLACE_OUTPUT_FILE) and os.path.exists(output_filepath):
        print("Output file exists, skipping.")
        continue


    prompt = PROMPT + text

    # check if prompt is in cache
    if not REPLACE:
        prompt_hash = llt.get_hash(prompt)
        cached_response = llt.check_response_cache(prompt_hash, response_db_conn)
        if cached_response:
            if MODE=='WRITE':
                response = cached_response[0]
                logprob = cached_response[1]
                perplexity = cached_response[2]
                parsed = json.loads(response.strip().removeprefix("```json").removesuffix("```").strip())
                with open(output_filepath, 'w') as f:
                    f.write(json.dumps(parsed, indent=2))
            continue

    #my_tokens = llt.count_tokens(prompt, MODEL)
    input_tokens += len(prompt)/3.5
    output_tokens += output_tokens_per_request
    n_requests += 1

    # add prompt to batch and check if batch is ready to submit
    if MODE=='BATCH':
        prompts_to_submit.append(prompt)
        if len(prompts_to_submit) >= REQUESTS_PER_BATCH:
            input_filename = f"summarize_agendas_batch_{batch_num}.jsonl"
            llt.create_chat_batch_job(prompts_to_submit, rel_path, input_filename, model=MODEL, batch_db_conn=batch_db_conn, response_db_conn=response_db_conn, overwrite=REPLACE, provider=LLM_PROVIDER)
            prompts_to_submit = []
            batch_num += 1
            print(f"    batch submitted")
    elif MODE=='WRITE':
        my_response = llt.get_response(prompt, MODEL, response_db_conn, overwrite=REPLACE, provider=LLM_PROVIDER)
        response = my_response['response']
        logprob = my_response['total_logprob']
        perplexity  = my_response['perplexity']
        n_direct += 1
        # Write response to file
        parsed = json.loads(response.strip().removeprefix("```json").removesuffix("```").strip())
        with open(output_filepath, 'w') as f:
            f.write(json.dumps(parsed, indent=2))

if (MODE=='BATCH') and (len(prompts_to_submit) > 0):
    input_filename = f"summarize_agendas_batch_{batch_num}.jsonl"
    llt.create_chat_batch_job(prompts_to_submit, rel_path, input_filename, model=MODEL, batch_db_conn=batch_db_conn, response_db_conn=response_db_conn, overwrite=REPLACE, provider=LLM_PROVIDER)
    prompts_to_submit = []
    print(f"    batch submitted")

t1 = time.time()
print(f"    elapsed time: {(t1-t0)/60:.2f} minutes, {n_direct:,} direct requests")

batch_db_conn.close()
response_db_conn.close()

print(f"{n_direct:,} direct requests made")

0 2018-05-10 ...
    elapsed time: 0.00 minutes, 0 direct requests
0 direct requests made


In [32]:
# Cost Estimate

total_input_cost = input_tokens / 1e6 * input_cost
total_output_cost = output_tokens / 1e6 * output_cost

print(f"Estimated number of requests: {n_requests:,.0f}")
print(f"Estimated total input tokens: {input_tokens:,.0f}")
print(f"Estimated total input cost: ${total_input_cost:.2f}")
print(f"Estimated total output tokens: {n_requests * output_tokens_per_request:,.0f}")
print(f"Estimated total output cost: ${total_output_cost:.2f}")
print(f"Estimated total cost: ${total_input_cost + total_output_cost:.2f}")

Estimated number of requests: 0
Estimated total input tokens: 0
Estimated total input cost: $0.00
Estimated total output tokens: 0
Estimated total output cost: $0.00
Estimated total cost: $0.00


In [35]:
print(json.dumps(parsed, indent=4))

{
    "agenda_items": [
        {
            "item_no": "1.",
            "title": "Director's Report and Commission Business",
            "address": "",
            "council_district": "",
            "consent_calendar": "no",
            "is_appeal": "n/a",
            "summary": "Update on City Planning Commission status reports and active assignments, legal actions and issues update, other items of interest, advance calendar, commission requests, and approval of meeting minutes from April 26, 2018. Includes note that Case No. CPC-2015-4703-VZC-ZV-SPR-ZAA-CU-CUB is continued to the June 14, 2018 meeting."
        },
        {
            "item_no": "2.",
            "title": "Neighborhood Council Presentation",
            "address": "",
            "council_district": "",
            "consent_calendar": "no",
            "is_appeal": "n/a",
            "summary": "Presentation by Neighborhood Council representatives on any Neighborhood Council resolution or community impact state